# Low-Stratum Document Investigation

## Purpose
Investigate the divergent behaviour between development and validation sets on low-stratum documents (1-2 accessions).

## Observation
- **Development:** Low precision (8.6%), high recall (72.7%) → Over-extraction
- **Validation:** High precision (90.9%), low recall (34.5%) → Under-extraction

## Investigation Steps
1. Generate document inventory for all low-stratum documents
2. Analyse error patterns (FP vs FN)
3. Compare document characteristics between splits
4. Identify specific problematic documents for manual review

---
## Setup and Configuration

In [2]:
import json
import os
import re
from pathlib import Path
from typing import Dict, List, Set, Optional, Tuple
from dataclasses import dataclass, field, asdict
from collections import Counter
import xml.etree.ElementTree as ET

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
# =============================================================================
# CONFIGURATION - UPDATE THESE PATHS
# =============================================================================

# Base directory
BASE_DIR = Path("/content/drive/MyDrive/AI6129")

# Input files
DEVELOPMENT_EVAL_FILE = BASE_DIR / "accession/investigation/claude-haiku-4.5_development_evaluation.json"
VALIDATION_EVAL_FILE = BASE_DIR / "accession/investigation/claude-haiku-4.5_validation_evaluation.json"
DEVELOPMENT_PRED_FILE = BASE_DIR / "accession/investigation/claude-haiku-4.5_development_predictions.json"
VALIDATION_PRED_FILE = BASE_DIR / "accession/investigation/claude-haiku-4.5_validation_predictions.json"
GROUND_TRUTH_FILE = BASE_DIR / "accession/validation_splits/validation_splits.json"
XML_FOLDER = BASE_DIR / "xml"

# Output directory
OUTPUT_DIR = BASE_DIR / "accession/investigation"

# Stratum definition
LOW_STRATUM_RANGE = (1, 2)  # Documents with 1-2 accessions

# Create output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")
print(f"ground truth directory : {GROUND_TRUTH_FILE}")
print(f"Output directory: {OUTPUT_DIR}")

Configuration loaded.
ground truth directory : /content/drive/MyDrive/AI6129/accession/validation_splits/validation_splits.json
Output directory: /content/drive/MyDrive/AI6129/accession/investigation


---
## Data Structures

In [4]:
@dataclass
class DocumentInventory:
    """Detailed inventory for a single document."""
    pmcid: str
    split: str
    stratum: str

    # Ground truth
    gt_count: int
    gt_accessions: List[str]

    # Predictions
    pred_count: int
    pred_accessions: List[str]

    # Analysis
    true_positives: List[str] = field(default_factory=list)
    false_positives: List[str] = field(default_factory=list)
    false_negatives: List[str] = field(default_factory=list)

    # Metrics
    precision: float = 0.0
    recall: float = 0.0
    f1: float = 0.0

    # Document characteristics (populated later)
    title: str = ""
    article_type: str = ""
    text_length: int = 0
    has_data_availability: bool = False
    has_tables: bool = False
    accession_locations: List[str] = field(default_factory=list)

    # Classification
    error_pattern: str = ""  # "over_extraction", "under_extraction", "balanced", "perfect"
    priority: str = ""  # "high", "medium", "low"

    def calculate_metrics(self):
        """Calculate precision, recall, F1 from TP/FP/FN."""
        tp = len(self.true_positives)
        fp = len(self.false_positives)
        fn = len(self.false_negatives)

        # Precision
        if tp + fp > 0:
            self.precision = tp / (tp + fp)
        else:
            self.precision = 1.0 if fn == 0 else 0.0

        # Recall
        if tp + fn > 0:
            self.recall = tp / (tp + fn)
        else:
            self.recall = 1.0

        # F1
        if self.precision + self.recall > 0:
            self.f1 = 2 * self.precision * self.recall / (self.precision + self.recall)
        else:
            self.f1 = 0.0

    def classify_error_pattern(self):
        """Classify the type of error pattern."""
        fp_count = len(self.false_positives)
        fn_count = len(self.false_negatives)

        if fp_count == 0 and fn_count == 0:
            self.error_pattern = "perfect"
            self.priority = "low"
        elif fp_count > fn_count * 2:
            self.error_pattern = "over_extraction"
            self.priority = "high" if fp_count > 5 else "medium"
        elif fn_count > fp_count * 2:
            self.error_pattern = "under_extraction"
            self.priority = "high" if fn_count > 1 else "medium"
        else:
            self.error_pattern = "balanced"
            self.priority = "medium" if (fp_count + fn_count) > 2 else "low"

---
## Data Loading Functions

In [18]:
def load_json_file(filepath: Path) -> dict:
    """Load a JSON file."""
    with open(filepath, 'r', encoding='utf-8') as f:
        return json.load(f)


def load_ground_truth(filepath: Path) -> Dict[str, dict]:
    """
    Load ground truth from validation_splits.json.

    Returns:
        Dictionary mapping PMCID -> {accessions, stratum, ...}
    """
    data = load_json_file(filepath)
    gt_lookup = {}

    # CORRECT KEY: document_details (not ground_truth)
    document_details = data.get("document_details", {})

    for pmcid, doc_info in document_details.items():
        accessions = doc_info.get("accessions", [])
        stratum = doc_info.get("stratum", "unknown")

        if accessions is None:
            accessions = []

        gt_lookup[pmcid] = {
            "accessions": accessions,
            "stratum": stratum,
            "count": len(accessions)
        }

    print(f"Loaded ground truth for {len(gt_lookup)} documents")


    stratum_counts = {}
    for pmcid, info in gt_lookup.items():
        stratum = info["stratum"]
        stratum_counts[stratum] = stratum_counts.get(stratum, 0) + 1
    print(f"  Stratum distribution: {stratum_counts}")

    return gt_lookup


def load_predictions(filepath: Path) -> Dict[str, List[str]]:
    """
    Load predictions from experiment output.

    Returns:
        Dictionary mapping PMCID -> list of extracted accessions
    """
    data = load_json_file(filepath)
    pred_lookup = {}

    for pred in data.get("predictions", []):
        pmcid = pred.get("pmcid", "")
        extracted = pred.get("extracted_accessions", [])

        if extracted is None:
            extracted = []

        pred_lookup[pmcid] = extracted

    print(f"Loaded predictions for {len(pred_lookup)} documents")
    return pred_lookup


def load_split_pmcids(filepath: Path, split_name: str) -> List[str]:
    """
    Load PMCIDs for a specific split.

    Returns:
        List of PMCIDs in the split
    """
    data = load_json_file(filepath)
    split_data = data.get("splits", {}).get(split_name, {})

    if isinstance(split_data, dict):
        pmcid_list = split_data.get("files", [])
    else:
        pmcid_list = split_data

    print(f"Loaded {len(pmcid_list)} PMCIDs for {split_name} split")
    return pmcid_list

---
## Analysis Functions

In [6]:
def normalise_accession(accession: str) -> str:
    """Normalise accession for comparison."""
    if accession is None:
        return ""

    normalised = str(accession).strip()

    # Remove version number
    if "." in normalised:
        normalised = normalised.split(".")[0]

    # Uppercase
    normalised = normalised.upper()

    return normalised


def analyse_document(pmcid: str,
                     split: str,
                     gt_data: dict,
                     predictions: List[str]) -> DocumentInventory:
    """
    Analyse a single document and create inventory.

    Args:
        pmcid: Document identifier
        split: Split name (development/validation)
        gt_data: Ground truth data dict
        predictions: List of predicted accessions

    Returns:
        DocumentInventory with analysis
    """
    # Get ground truth
    gt_accessions = gt_data.get("accessions", [])
    stratum = gt_data.get("stratum", "unknown")

    # Normalise for comparison
    gt_normalised = set(normalise_accession(acc) for acc in gt_accessions if acc)
    pred_normalised = set(normalise_accession(acc) for acc in predictions if acc)

    # Calculate sets
    true_positives = gt_normalised.intersection(pred_normalised)
    false_positives = pred_normalised - gt_normalised
    false_negatives = gt_normalised - pred_normalised

    # Create inventory
    inventory = DocumentInventory(
        pmcid=pmcid,
        split=split,
        stratum=stratum,
        gt_count=len(gt_accessions),
        gt_accessions=gt_accessions,
        pred_count=len(predictions),
        pred_accessions=predictions,
        true_positives=sorted(list(true_positives)),
        false_positives=sorted(list(false_positives)),
        false_negatives=sorted(list(false_negatives))
    )

    # Calculate metrics and classify
    inventory.calculate_metrics()
    inventory.classify_error_pattern()

    return inventory


def filter_low_stratum(inventories: List[DocumentInventory]) -> List[DocumentInventory]:
    """Filter to only low-stratum documents."""
    return [inv for inv in inventories if inv.stratum == "low"]

---
## XML Analysis Functions

In [7]:
def find_xml_file(pmcid: str, xml_folder: Path) -> Optional[Path]:
    """
    Find XML file for a given PMCID.

    Handles patterns like:
    - PMC1234567.xml
    - PMC1234567_20241014.xml
    """
    # Try exact match first
    exact_path = xml_folder / f"{pmcid}.xml"
    if exact_path.exists():
        return exact_path

    # Try pattern match
    pattern = f"{pmcid}*.xml"
    matches = list(xml_folder.glob(pattern))

    if matches:
        return matches[0]

    return None


def extract_document_characteristics(xml_path: Path) -> dict:
    """
    Extract characteristics from XML file.

    Returns:
        Dictionary with title, article_type, text_length, etc.
    """
    characteristics = {
        "title": "",
        "article_type": "",
        "text_length": 0,
        "has_data_availability": False,
        "has_tables": False,
        "section_count": 0,
        "table_count": 0
    }

    try:
        with open(xml_path, 'r', encoding='utf-8') as f:
            content = f.read()

        characteristics["text_length"] = len(content)

        # Parse XML
        tree = ET.parse(xml_path)
        root = tree.getroot()

        # Extract title
        title_elem = root.find(".//article-title")
        if title_elem is not None and title_elem.text:
            characteristics["title"] = title_elem.text[:200]  # Truncate

        # Extract article type
        article_type_elem = root.find(".//article-categories//subject")
        if article_type_elem is not None and article_type_elem.text:
            characteristics["article_type"] = article_type_elem.text

        # Check for data availability section
        content_lower = content.lower()
        data_avail_patterns = [
            "data availability",
            "data deposition",
            "accession number",
            "genbank accession",
            "sequence data"
        ]
        characteristics["has_data_availability"] = any(
            pattern in content_lower for pattern in data_avail_patterns
        )

        # Count tables
        tables = root.findall(".//table-wrap")
        characteristics["table_count"] = len(tables)
        characteristics["has_tables"] = len(tables) > 0

        # Count sections
        sections = root.findall(".//sec")
        characteristics["section_count"] = len(sections)

    except Exception as e:
        print(f"  Warning: Could not parse XML: {e}")

    return characteristics


def find_accession_locations(xml_path: Path, accessions: List[str]) -> List[str]:
    """
    Find where specific accessions appear in the XML.

    Returns:
        List of location descriptions
    """
    locations = []

    if not accessions:
        return locations

    try:
        with open(xml_path, 'r', encoding='utf-8') as f:
            content = f.read()

        content_lower = content.lower()

        # Define section markers
        section_markers = [
            ("data availability", "Data Availability"),
            ("methods", "Methods"),
            ("materials and methods", "Methods"),
            ("results", "Results"),
            ("abstract", "Abstract"),
            ("<table-wrap", "Table"),
            ("supplementary", "Supplementary"),
            ("references", "References")
        ]

        for acc in accessions:
            acc_lower = acc.lower()

            # Find position in content
            pos = content_lower.find(acc_lower)

            if pos == -1:
                locations.append(f"{acc}: NOT FOUND")
                continue

            # Determine section by looking backwards
            section_found = "Unknown Section"
            for marker, section_name in section_markers:
                marker_pos = content_lower.rfind(marker, 0, pos)
                if marker_pos != -1:
                    section_found = section_name
                    break

            locations.append(f"{acc}: {section_found}")

    except Exception as e:
        locations.append(f"Error: {e}")

    return locations

---
## Main Processing

In [8]:
def generate_inventory(gt_lookup: Dict[str, dict],
                       dev_predictions: Dict[str, List[str]],
                       val_predictions: Dict[str, List[str]],
                       dev_pmcids: List[str],
                       val_pmcids: List[str]) -> List[DocumentInventory]:
    """
    Generate inventory for all documents.

    Returns:
        List of DocumentInventory objects
    """
    all_inventories = []

    # Process development split
    print("\nProcessing development split...")
    for pmcid in dev_pmcids:
        gt_data = gt_lookup.get(pmcid, {"accessions": [], "stratum": "unknown"})
        predictions = dev_predictions.get(pmcid, [])

        inventory = analyse_document(pmcid, "development", gt_data, predictions)
        all_inventories.append(inventory)

    # Process validation split
    print("Processing validation split...")
    for pmcid in val_pmcids:
        gt_data = gt_lookup.get(pmcid, {"accessions": [], "stratum": "unknown"})
        predictions = val_predictions.get(pmcid, [])

        inventory = analyse_document(pmcid, "validation", gt_data, predictions)
        all_inventories.append(inventory)

    print(f"\nTotal documents processed: {len(all_inventories)}")
    return all_inventories

In [9]:
def enrich_with_xml_data(inventories: List[DocumentInventory],
                         xml_folder: Path) -> None:
    """
    Enrich inventory with XML document characteristics.
    Modifies inventories in place.
    """
    print("\nEnriching with XML data...")

    for i, inv in enumerate(inventories):
        xml_path = find_xml_file(inv.pmcid, xml_folder)

        if xml_path is None:
            print(f"  Warning: XML not found for {inv.pmcid}")
            continue

        # Extract characteristics
        chars = extract_document_characteristics(xml_path)
        inv.title = chars["title"]
        inv.article_type = chars["article_type"]
        inv.text_length = chars["text_length"]
        inv.has_data_availability = chars["has_data_availability"]
        inv.has_tables = chars["has_tables"]

        # Find accession locations for FN (missed accessions)
        if inv.false_negatives:
            inv.accession_locations = find_accession_locations(
                xml_path, inv.false_negatives
            )

        # Progress
        if (i + 1) % 10 == 0:
            print(f"  Processed {i + 1}/{len(inventories)} documents")

    print("XML enrichment complete.")

---
## Output Generation

In [10]:
def generate_summary_report(low_stratum: List[DocumentInventory]) -> str:
    """
    Generate text summary report.
    """
    lines = []
    lines.append("=" * 80)
    lines.append("LOW-STRATUM INVESTIGATION SUMMARY")
    lines.append("=" * 80)

    # Split breakdown
    dev_docs = [d for d in low_stratum if d.split == "development"]
    val_docs = [d for d in low_stratum if d.split == "validation"]

    lines.append(f"\nTotal low-stratum documents: {len(low_stratum)}")
    lines.append(f"  Development: {len(dev_docs)}")
    lines.append(f"  Validation: {len(val_docs)}")

    # Error pattern breakdown
    lines.append("\n" + "-" * 40)
    lines.append("ERROR PATTERN BREAKDOWN")
    lines.append("-" * 40)

    for split_name, docs in [("Development", dev_docs), ("Validation", val_docs)]:
        lines.append(f"\n{split_name}:")
        patterns = Counter(d.error_pattern for d in docs)
        for pattern, count in patterns.most_common():
            lines.append(f"  {pattern}: {count}")

    # Aggregate metrics
    lines.append("\n" + "-" * 40)
    lines.append("AGGREGATE METRICS")
    lines.append("-" * 40)

    for split_name, docs in [("Development", dev_docs), ("Validation", val_docs)]:
        total_tp = sum(len(d.true_positives) for d in docs)
        total_fp = sum(len(d.false_positives) for d in docs)
        total_fn = sum(len(d.false_negatives) for d in docs)

        micro_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
        micro_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0

        lines.append(f"\n{split_name}:")
        lines.append(f"  Total TP: {total_tp}")
        lines.append(f"  Total FP: {total_fp}")
        lines.append(f"  Total FN: {total_fn}")
        lines.append(f"  Micro Precision: {micro_precision:.3f}")
        lines.append(f"  Micro Recall: {micro_recall:.3f}")

    # High priority documents
    lines.append("\n" + "-" * 40)
    lines.append("HIGH PRIORITY DOCUMENTS FOR REVIEW")
    lines.append("-" * 40)

    high_priority = [d for d in low_stratum if d.priority == "high"]
    high_priority.sort(key=lambda x: len(x.false_positives) + len(x.false_negatives), reverse=True)

    for doc in high_priority[:10]:
        lines.append(f"\n{doc.pmcid} ({doc.split}):")
        lines.append(f"  Pattern: {doc.error_pattern}")
        lines.append(f"  GT: {doc.gt_count}, Pred: {doc.pred_count}")
        lines.append(f"  TP: {len(doc.true_positives)}, FP: {len(doc.false_positives)}, FN: {len(doc.false_negatives)}")
        if doc.false_positives:
            lines.append(f"  False Positives: {', '.join(doc.false_positives[:5])}{'...' if len(doc.false_positives) > 5 else ''}")
        if doc.false_negatives:
            lines.append(f"  False Negatives: {', '.join(doc.false_negatives)}")

    lines.append("\n" + "=" * 80)

    return "\n".join(lines)

In [11]:
def generate_csv_output(inventories: List[DocumentInventory],
                        output_path: Path) -> None:
    """
    Generate CSV output for detailed analysis.
    """
    import csv

    fieldnames = [
        "pmcid", "split", "stratum", "error_pattern", "priority",
        "gt_count", "pred_count", "tp_count", "fp_count", "fn_count",
        "precision", "recall", "f1",
        "gt_accessions", "pred_accessions",
        "true_positives", "false_positives", "false_negatives",
        "title", "article_type", "text_length",
        "has_data_availability", "has_tables",
        "accession_locations"
    ]

    with open(output_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        for inv in inventories:
            row = {
                "pmcid": inv.pmcid,
                "split": inv.split,
                "stratum": inv.stratum,
                "error_pattern": inv.error_pattern,
                "priority": inv.priority,
                "gt_count": inv.gt_count,
                "pred_count": inv.pred_count,
                "tp_count": len(inv.true_positives),
                "fp_count": len(inv.false_positives),
                "fn_count": len(inv.false_negatives),
                "precision": f"{inv.precision:.3f}",
                "recall": f"{inv.recall:.3f}",
                "f1": f"{inv.f1:.3f}",
                "gt_accessions": "; ".join(inv.gt_accessions),
                "pred_accessions": "; ".join(inv.pred_accessions[:20]),
                "true_positives": "; ".join(inv.true_positives),
                "false_positives": "; ".join(inv.false_positives[:20]),
                "false_negatives": "; ".join(inv.false_negatives),
                "title": inv.title[:100] if inv.title else "",
                "article_type": inv.article_type,
                "text_length": inv.text_length,
                "has_data_availability": inv.has_data_availability,
                "has_tables": inv.has_tables,
                "accession_locations": "; ".join(inv.accession_locations)
            }
            writer.writerow(row)

    print(f"CSV saved to: {output_path}")

In [12]:
def generate_comparison_table(low_stratum: List[DocumentInventory]) -> str:
    """
    Generate comparison table between splits.
    """
    dev_docs = [d for d in low_stratum if d.split == "development"]
    val_docs = [d for d in low_stratum if d.split == "validation"]

    def calc_stats(docs):
        if not docs:
            return {}

        # Article types
        article_types = Counter(d.article_type for d in docs if d.article_type)

        # Average text length
        text_lengths = [d.text_length for d in docs if d.text_length > 0]
        avg_length = sum(text_lengths) / len(text_lengths) if text_lengths else 0

        # Data availability
        has_da = sum(1 for d in docs if d.has_data_availability)

        # Tables
        has_tables = sum(1 for d in docs if d.has_tables)

        # Error patterns
        over_extract = sum(1 for d in docs if d.error_pattern == "over_extraction")
        under_extract = sum(1 for d in docs if d.error_pattern == "under_extraction")

        return {
            "count": len(docs),
            "article_types": article_types.most_common(3),
            "avg_text_length": avg_length,
            "has_data_availability": has_da,
            "has_tables": has_tables,
            "over_extraction": over_extract,
            "under_extraction": under_extract
        }

    dev_stats = calc_stats(dev_docs)
    val_stats = calc_stats(val_docs)

    lines = []
    lines.append("\n" + "=" * 80)
    lines.append("SPLIT COMPARISON TABLE")
    lines.append("=" * 80)
    lines.append(f"\n{'Characteristic':<30} {'Development':<20} {'Validation':<20}")
    lines.append("-" * 70)
    lines.append(f"{'Document Count':<30} {dev_stats.get('count', 0):<20} {val_stats.get('count', 0):<20}")
    lines.append(f"{'Avg Text Length':<30} {dev_stats.get('avg_text_length', 0):,.0f}{'':<13} {val_stats.get('avg_text_length', 0):,.0f}")
    lines.append(f"{'Has Data Availability':<30} {dev_stats.get('has_data_availability', 0):<20} {val_stats.get('has_data_availability', 0):<20}")
    lines.append(f"{'Has Tables':<30} {dev_stats.get('has_tables', 0):<20} {val_stats.get('has_tables', 0):<20}")
    lines.append(f"{'Over-extraction Pattern':<30} {dev_stats.get('over_extraction', 0):<20} {val_stats.get('over_extraction', 0):<20}")
    lines.append(f"{'Under-extraction Pattern':<30} {dev_stats.get('under_extraction', 0):<20} {val_stats.get('under_extraction', 0):<20}")

    lines.append("\nTop Article Types:")
    lines.append(f"  Development: {dev_stats.get('article_types', [])}")
    lines.append(f"  Validation: {val_stats.get('article_types', [])}")

    return "\n".join(lines)

---
## Execute Investigation

In [19]:
# =============================================================================
# STEP 1: Load all data
# =============================================================================

print("=" * 80)
print("LOADING DATA")
print("=" * 80)

# Load ground truth
gt_lookup = load_ground_truth(GROUND_TRUTH_FILE)

# Load predictions
dev_predictions = load_predictions(DEVELOPMENT_PRED_FILE)
val_predictions = load_predictions(VALIDATION_PRED_FILE)

# Load split PMCIDs
dev_pmcids = load_split_pmcids(GROUND_TRUTH_FILE, "development")
val_pmcids = load_split_pmcids(GROUND_TRUTH_FILE, "validation")

print("\nData loading complete.")

LOADING DATA
Loaded ground truth for 227 documents
  Stratum distribution: {'zero': 59, 'low': 41, 'high': 80, 'medium': 47}
Loaded predictions for 44 documents
Loaded predictions for 116 documents
Loaded 44 PMCIDs for development split
Loaded 116 PMCIDs for validation split

Data loading complete.


In [20]:
# =============================================================================
# STEP 2: Generate inventory for all documents
# =============================================================================

print("\n" + "=" * 80)
print("GENERATING INVENTORY")
print("=" * 80)

all_inventories = generate_inventory(
    gt_lookup,
    dev_predictions,
    val_predictions,
    dev_pmcids,
    val_pmcids
)


GENERATING INVENTORY

Processing development split...
Processing validation split...

Total documents processed: 160


In [21]:
# =============================================================================
# STEP 3: Filter to low-stratum only
# =============================================================================

print("\n" + "=" * 80)
print("FILTERING LOW-STRATUM")
print("=" * 80)

low_stratum = filter_low_stratum(all_inventories)

print(f"\nLow-stratum documents: {len(low_stratum)}")
print(f"  Development: {len([d for d in low_stratum if d.split == 'development'])}")
print(f"  Validation: {len([d for d in low_stratum if d.split == 'validation'])}")


FILTERING LOW-STRATUM

Low-stratum documents: 29
  Development: 8
  Validation: 21


In [22]:
# =============================================================================
# STEP 4: Enrich with XML data
# =============================================================================

print("\n" + "=" * 80)
print("ENRICHING WITH XML DATA")
print("=" * 80)

enrich_with_xml_data(low_stratum, XML_FOLDER)


ENRICHING WITH XML DATA

Enriching with XML data...
  Processed 10/29 documents
  Processed 20/29 documents
XML enrichment complete.


In [23]:
# =============================================================================
# STEP 5: Generate and display summary report
# =============================================================================

print("\n")
summary_report = generate_summary_report(low_stratum)
print(summary_report)

# Save to file
summary_path = OUTPUT_DIR / "low_stratum_summary.txt"
with open(summary_path, 'w', encoding='utf-8') as f:
    f.write(summary_report)
print(f"\nSummary saved to: {summary_path}")



LOW-STRATUM INVESTIGATION SUMMARY

Total low-stratum documents: 29
  Development: 8
  Validation: 21

----------------------------------------
ERROR PATTERN BREAKDOWN
----------------------------------------

Development:
  under_extraction: 3
  perfect: 3
  over_extraction: 2

Validation:
  under_extraction: 16
  perfect: 4
  balanced: 1

----------------------------------------
AGGREGATE METRICS
----------------------------------------

Development:
  Total TP: 8
  Total FP: 85
  Total FN: 3
  Micro Precision: 0.086
  Micro Recall: 0.727

Validation:
  Total TP: 10
  Total FP: 1
  Total FN: 19
  Micro Precision: 0.909
  Micro Recall: 0.345

----------------------------------------
HIGH PRIORITY DOCUMENTS FOR REVIEW
----------------------------------------

PMC6935380 (development):
  Pattern: over_extraction
  GT: 2, Pred: 80
  TP: 2, FP: 78, FN: 0
  False Positives: MH548441, MH548442, MH548443, MH548444, MH548445...

PMC7764154 (development):
  Pattern: over_extraction
  GT: 2, P

In [24]:
# =============================================================================
# STEP 6: Generate comparison table
# =============================================================================

comparison = generate_comparison_table(low_stratum)
print(comparison)


SPLIT COMPARISON TABLE

Characteristic                 Development          Validation          
----------------------------------------------------------------------
Document Count                 8                    21                  
Avg Text Length                111,559              117,767
Has Data Availability          4                    13                  
Has Tables                     7                    15                  
Over-extraction Pattern        2                    0                   
Under-extraction Pattern       3                    16                  

Top Article Types:
  Development: [('Article', 3), ('Research Article', 1), ('Prokaryotes', 1)]
  Validation: [('Article', 6), ('Research Article', 5), ('Case Report', 5)]


In [25]:
# =============================================================================
# STEP 7: Save detailed CSV
# =============================================================================

csv_path = OUTPUT_DIR / "low_stratum_inventory.csv"
generate_csv_output(low_stratum, csv_path)

CSV saved to: /content/drive/MyDrive/AI6129/accession/investigation/low_stratum_inventory.csv


---
## Detailed Document Analysis

In [26]:
# =============================================================================
# DEVELOPMENT SPLIT ANALYSIS
# =============================================================================

print("=" * 80)
print("DEVELOPMENT SPLIT - DETAILED ANALYSIS")
print("=" * 80)

dev_low = [d for d in low_stratum if d.split == "development"]
dev_low.sort(key=lambda x: len(x.false_positives), reverse=True)

print(f"\nDocuments sorted by false positive count:\n")

for doc in dev_low:
    print(f"{doc.pmcid}:")
    print(f"  Ground Truth ({doc.gt_count}): {doc.gt_accessions}")
    print(f"  Predicted ({doc.pred_count}): {doc.pred_accessions[:10]}{'...' if len(doc.pred_accessions) > 10 else ''}")
    print(f"  TP: {len(doc.true_positives)}, FP: {len(doc.false_positives)}, FN: {len(doc.false_negatives)}")
    print(f"  Precision: {doc.precision:.3f}, Recall: {doc.recall:.3f}")
    print(f"  Pattern: {doc.error_pattern}")
    if doc.false_positives:
        print(f"  FALSE POSITIVES: {doc.false_positives[:10]}")
    if doc.false_negatives:
        print(f"  FALSE NEGATIVES: {doc.false_negatives}")
    print()

DEVELOPMENT SPLIT - DETAILED ANALYSIS

Documents sorted by false positive count:

PMC6935380:
  Ground Truth (2): ['MH548440', 'MH548519']
  Predicted (80): ['MH548440', 'MH548441', 'MH548442', 'MH548443', 'MH548444', 'MH548445', 'MH548446', 'MH548447', 'MH548448', 'MH548449']...
  TP: 2, FP: 78, FN: 0
  Precision: 0.025, Recall: 1.000
  Pattern: over_extraction
  FALSE POSITIVES: ['MH548441', 'MH548442', 'MH548443', 'MH548444', 'MH548445', 'MH548446', 'MH548447', 'MH548448', 'MH548449', 'MH548450']

PMC7764154:
  Ground Truth (2): ['MW006474', 'MW006482']
  Predicted (9): ['MW006474', 'MW006475', 'MW006476', 'MW006477', 'MW006478', 'MW006479', 'MW006480', 'MW006481', 'MW006482']
  TP: 2, FP: 7, FN: 0
  Precision: 0.222, Recall: 1.000
  Pattern: over_extraction
  FALSE POSITIVES: ['MW006475', 'MW006476', 'MW006477', 'MW006478', 'MW006479', 'MW006480', 'MW006481']

PMC5075293:
  Ground Truth (1): ['P11101']
  Predicted (0): []
  TP: 0, FP: 0, FN: 1
  Precision: 0.000, Recall: 0.000
  Pa

In [27]:
# =============================================================================
# VALIDATION SPLIT ANALYSIS
# =============================================================================

print("=" * 80)
print("VALIDATION SPLIT - DETAILED ANALYSIS")
print("=" * 80)

val_low = [d for d in low_stratum if d.split == "validation"]
val_low.sort(key=lambda x: len(x.false_negatives), reverse=True)

print(f"\nDocuments sorted by false negative count:\n")

for doc in val_low:
    print(f"{doc.pmcid}:")
    print(f"  Ground Truth ({doc.gt_count}): {doc.gt_accessions}")
    print(f"  Predicted ({doc.pred_count}): {doc.pred_accessions[:10]}{'...' if len(doc.pred_accessions) > 10 else ''}")
    print(f"  TP: {len(doc.true_positives)}, FP: {len(doc.false_positives)}, FN: {len(doc.false_negatives)}")
    print(f"  Precision: {doc.precision:.3f}, Recall: {doc.recall:.3f}")
    print(f"  Pattern: {doc.error_pattern}")
    if doc.false_negatives:
        print(f"  FALSE NEGATIVES: {doc.false_negatives}")
        if doc.accession_locations:
            print(f"  LOCATIONS: {doc.accession_locations}")
    if doc.false_positives:
        print(f"  FALSE POSITIVES: {doc.false_positives[:5]}")
    print()

VALIDATION SPLIT - DETAILED ANALYSIS

Documents sorted by false negative count:

PMC7694379:
  Ground Truth (2): ['AP001918', 'FD006229']
  Predicted (1): ['SRR1646144']
  TP: 0, FP: 1, FN: 2
  Precision: 0.000, Recall: 0.000
  Pattern: balanced
  FALSE NEGATIVES: ['AP001918', 'FD006229']
  LOCATIONS: ['AP001918: Methods', 'FD006229: Methods']
  FALSE POSITIVES: ['SRR1646144']

PMC8101457:
  Ground Truth (2): ['MW198823', 'MW199053']
  Predicted (0): []
  TP: 0, FP: 0, FN: 2
  Precision: 0.000, Recall: 0.000
  Pattern: under_extraction
  FALSE NEGATIVES: ['MW198823', 'MW199053']
  LOCATIONS: ['MW198823: Methods', 'MW199053: Methods']

PMC8909193:
  Ground Truth (1): ['PMC90463']
  Predicted (0): []
  TP: 0, FP: 0, FN: 1
  Precision: 0.000, Recall: 0.000
  Pattern: under_extraction
  FALSE NEGATIVES: ['PMC90463']
  LOCATIONS: ['PMC90463: Data Availability']

PMC9603869:
  Ground Truth (2): ['PRJNA751163', 'PMC94128']
  Predicted (1): ['PRJNA751163']
  TP: 1, FP: 0, FN: 1
  Precision: 1.

---
## False Positive Pattern Analysis

In [28]:
# =============================================================================
# ANALYSE FALSE POSITIVE PATTERNS
# =============================================================================

print("=" * 80)
print("FALSE POSITIVE PATTERN ANALYSIS")
print("=" * 80)

# Collect all false positives
all_fps = []
for doc in low_stratum:
    for fp in doc.false_positives:
        all_fps.append({
            "accession": fp,
            "pmcid": doc.pmcid,
            "split": doc.split
        })

print(f"\nTotal false positives: {len(all_fps)}")

# Analyse patterns
fp_patterns = Counter()

for fp_info in all_fps:
    fp = fp_info["accession"]

    # Classify by pattern
    if re.match(r'^[A-Z]{2}\d{6}$', fp):
        fp_patterns["Standard nucleotide (XX######)"] += 1
    elif re.match(r'^[A-Z]{3}\d{5}$', fp):
        fp_patterns["Protein (XXX#####)"] += 1
    elif re.match(r'^[A-Z]{4}\d{8,}$', fp):
        fp_patterns["WGS (XXXX########)"] += 1
    elif re.match(r'^SRR\d+$', fp):
        fp_patterns["SRA Run (SRR...)"] += 1
    elif re.match(r'^PRJNA\d+$', fp):
        fp_patterns["BioProject (PRJNA...)"] += 1
    elif re.match(r'^SAMN\d+$', fp):
        fp_patterns["BioSample (SAMN...)"] += 1
    elif re.match(r'^NC_\d+$', fp):
        fp_patterns["RefSeq (NC_...)"] += 1
    elif re.match(r'^[A-Z]{1,2}\d+$', fp):
        fp_patterns["Short format (X#...)"] += 1
    else:
        fp_patterns["Other"] += 1

print("\nFalse Positive Patterns:")
for pattern, count in fp_patterns.most_common():
    print(f"  {pattern}: {count}")

# Show examples of "Other" category
other_fps = [fp["accession"] for fp in all_fps if not any([
    re.match(r'^[A-Z]{2}\d{6}$', fp["accession"]),
    re.match(r'^[A-Z]{3}\d{5}$', fp["accession"]),
    re.match(r'^[A-Z]{4}\d{8,}$', fp["accession"]),
    re.match(r'^SRR\d+$', fp["accession"]),
    re.match(r'^PRJNA\d+$', fp["accession"]),
    re.match(r'^SAMN\d+$', fp["accession"]),
    re.match(r'^NC_\d+$', fp["accession"]),
    re.match(r'^[A-Z]{1,2}\d+$', fp["accession"])
])]

if other_fps:
    print(f"\n'Other' pattern examples: {other_fps[:20]}")

FALSE POSITIVE PATTERN ANALYSIS

Total false positives: 86

False Positive Patterns:
  Standard nucleotide (XX######): 85
  SRA Run (SRR...): 1


---
## False Negative Location Analysis

In [29]:
# =============================================================================
# ANALYSE FALSE NEGATIVE LOCATIONS
# =============================================================================

print("=" * 80)
print("FALSE NEGATIVE LOCATION ANALYSIS")
print("=" * 80)

# Collect all locations
location_counter = Counter()

for doc in low_stratum:
    for loc in doc.accession_locations:
        # Extract section name
        if ": " in loc:
            section = loc.split(": ")[1]
            location_counter[section] += 1

print("\nFalse Negative Locations:")
for location, count in location_counter.most_common():
    print(f"  {location}: {count}")

# Documents with FN in specific sections
print("\nDocuments with FN in 'NOT FOUND' (potential GT issues):")
for doc in low_stratum:
    not_found = [loc for loc in doc.accession_locations if "NOT FOUND" in loc]
    if not_found:
        print(f"  {doc.pmcid} ({doc.split}): {not_found}")

FALSE NEGATIVE LOCATION ANALYSIS

False Negative Locations:
  Methods: 14
  Data Availability: 5
  Unknown Section: 2
  Abstract: 1

Documents with FN in 'NOT FOUND' (potential GT issues):


---
## Conclusions and Next Steps

In [30]:
# =============================================================================
# GENERATE CONCLUSIONS
# =============================================================================

print("=" * 80)
print("INVESTIGATION CONCLUSIONS")
print("=" * 80)

dev_low = [d for d in low_stratum if d.split == "development"]
val_low = [d for d in low_stratum if d.split == "validation"]

# Calculate key differences
dev_over = sum(1 for d in dev_low if d.error_pattern == "over_extraction")
dev_under = sum(1 for d in dev_low if d.error_pattern == "under_extraction")
val_over = sum(1 for d in val_low if d.error_pattern == "over_extraction")
val_under = sum(1 for d in val_low if d.error_pattern == "under_extraction")

print(f"""
KEY FINDINGS:

1. ERROR PATTERN DISTRIBUTION:
   Development: {dev_over} over-extraction, {dev_under} under-extraction
   Validation:  {val_over} over-extraction, {val_under} under-extraction

2. HYPOTHESIS CHECK:
   - If development has more over-extraction: Documents may have more
     accession-like patterns that aren't true accessions
   - If validation has more under-extraction: Model may be too conservative
     or accessions are in unusual locations

3. RECOMMENDED ACTIONS:
   a) Review documents with 'NOT FOUND' false negatives - potential GT issues
   b) Examine false positive patterns for systematic errors
   c) Check if article types differ between splits
   d) Verify ground truth for all low-stratum documents

4. FILES GENERATED:
   - {OUTPUT_DIR / 'low_stratum_summary.txt'}
   - {OUTPUT_DIR / 'low_stratum_inventory.csv'}
""")

INVESTIGATION CONCLUSIONS

KEY FINDINGS:

1. ERROR PATTERN DISTRIBUTION:
   Development: 2 over-extraction, 3 under-extraction
   Validation:  0 over-extraction, 16 under-extraction
   
2. HYPOTHESIS CHECK:
   - If development has more over-extraction: Documents may have more
     accession-like patterns that aren't true accessions
   - If validation has more under-extraction: Model may be too conservative
     or accessions are in unusual locations

3. RECOMMENDED ACTIONS:
   a) Review documents with 'NOT FOUND' false negatives - potential GT issues
   b) Examine false positive patterns for systematic errors
   c) Check if article types differ between splits
   d) Verify ground truth for all low-stratum documents

4. FILES GENERATED:
   - /content/drive/MyDrive/AI6129/accession/investigation/low_stratum_summary.txt
   - /content/drive/MyDrive/AI6129/accession/investigation/low_stratum_inventory.csv



In [31]:
# =============================================================================
# PRIORITY ACTION LIST
# =============================================================================

print("=" * 80)
print("PRIORITY ACTION LIST")
print("=" * 80)

# Documents that need immediate attention
high_priority = [d for d in low_stratum if d.priority == "high"]
high_priority.sort(key=lambda x: (x.split, -len(x.false_positives) - len(x.false_negatives)))

print(f"\nHigh priority documents requiring manual review ({len(high_priority)}):")
print()

for doc in high_priority:
    action = ""
    if doc.error_pattern == "over_extraction":
        action = "Examine FPs - are they valid accessions missing from GT?"
    elif doc.error_pattern == "under_extraction":
        action = "Examine FNs - where are they in the document?"

    print(f"[ ] {doc.pmcid} ({doc.split})")
    print(f"    Pattern: {doc.error_pattern}")
    print(f"    GT: {doc.gt_accessions}")
    print(f"    Action: {action}")
    print()

PRIORITY ACTION LIST

High priority documents requiring manual review (3):

[ ] PMC6935380 (development)
    Pattern: over_extraction
    GT: ['MH548440', 'MH548519']
    Action: Examine FPs - are they valid accessions missing from GT?

[ ] PMC7764154 (development)
    Pattern: over_extraction
    GT: ['MW006474', 'MW006482']
    Action: Examine FPs - are they valid accessions missing from GT?

[ ] PMC8101457 (validation)
    Pattern: under_extraction
    GT: ['MW198823', 'MW199053']
    Action: Examine FNs - where are they in the document?

